# AutoGaze Master Inference Suite (v2.0)

This notebook is the complete, interactive hub for testing all AutoGaze model combinations (NVILA, Qwen-VL, V-JEPA2).

### What you can do:
1.  **Compare Backbones**: Test how SigLIP vs V-JEPA2 affects accuracy and speed.
2.  **Measure Efficiency**: Profile VRAM and Latency for AutoGaze ON vs OFF.
3.  **Visualize Gaze**: See the 14x14 attention heatmap on top of your video.
4.  **Sweep Ratios**: Find the optimal balance between token count and quality.


## 1. Setup Environment

In [ ]:
import sys, os, time, torch, cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from autogaze.eval.models import load_runner, RUNNERS

# Ensure we are in the notebook directory, but can access project root
if not os.getcwd().endswith('notebooks'):
    os.chdir('notebooks')

REPO_ROOT = Path(os.getcwd()).parent
AG_PATH   = str(REPO_ROOT / "weights/AutoGaze")
DEVICE    = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

def _sync():
    if DEVICE == "cuda": torch.cuda.synchronize()
    elif DEVICE == "mps": torch.mps.synchronize()

print(f"Device: {DEVICE}")
print(f"Project Root: {REPO_ROOT}")
print(f"Available Runners: {sorted(RUNNERS.keys())}")

## 2. Load Video & Config

In [ ]:
VIDEO_PATH = REPO_ROOT / "assets/example_input.mp4"
QUESTION   = "이 비디오에서 무엇이 일어나고 있나요? (What is happening in this video?)"
RATIO      = 0.25  # Budget: 25% of tokens

def load_video_frames(path, n=16):
    cap = cv2.VideoCapture(str(path))
    frames = []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total-1, n, dtype=int)
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret: frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames

frames = load_video_frames(VIDEO_PATH)
plt.imshow(frames[0])
plt.title(f"Loaded {len(frames)} frames")
plt.axis('off')
plt.show()

## 3. Benchmarking: SigLIP (NVILA) vs V-JEPA2 Substitute

In [ ]:
def benchmark_full_pipeline(runner_key, model_path, use_ag=True, **kwargs):
    print(f"\n--- {runner_key.upper()} (AG={'ON' if use_ag else 'OFF'}) ---")
    
    ag_p = AG_PATH if use_ag else None
    
    # 1. Load Model
    t0 = time.perf_counter()
    runner = load_runner(
        mllm=runner_key,
        model_path=str(model_path),
        autogaze_path=ag_p,
        gazing_ratio=RATIO,
        **kwargs
    )
    print(f"Model loaded in {time.perf_counter()-t0:.1f}s")
    
    # 2. Warm up
    _ = runner.run(frames, QUESTION)
    
    # 3. Timed Inference
    _sync()
    t0 = time.perf_counter()
    answer = runner.run(frames, QUESTION)
    _sync()
    latency = (time.perf_counter() - t0) * 1000
    
    # 4. Memory Profiling (CUDA only)
    vram = 0
    if torch.cuda.is_available():
        vram = torch.cuda.max_memory_allocated() / 1024**3
        
    print(f"Answer : {answer}")
    print(f"Latency: {latency:.1f} ms")
    if vram > 0: print(f"VRAM   : {vram:.2f} GB")
    
    return {"latency": latency, "vram": vram, "answer": answer, "runner": runner}

# Run Comparison (Uncomment to execute - requires ~20GB VRAM total)
# res_nvila = benchmark_full_pipeline("nvila", REPO_ROOT / "weights/NVILA-8B-HD-Video")
# res_vjepa = benchmark_full_pipeline("nvila_vjepa2", 
#                                     model_path=REPO_ROOT / "weights/NVILA-8B-HD-Video",
#                                     vjepa2_path=REPO_ROOT / "weights/vjepa2-vitl-fpc64-256")

## 4. Visualizing the Gaze Map Evolution

In [ ]:
import torch.nn.functional as F

def plot_gaze_sweep(runner, frames, ratios=[0.1, 0.25, 0.5, 1.0]):
    if not hasattr(runner, '_run_autogaze'):
        print("This runner doesn't support gaze visualization.")
        return
        
    T = min(len(frames), 4)
    fig, axes = plt.subplots(len(ratios), T, figsize=(T*3, len(ratios)*3))
    
    orig_ratio = runner.gazing_ratio
    
    for row, r in enumerate(ratios):
        if runner.selector: runner.selector.gazing_ratio = r
        else: runner.processor.gazing_ratio_tile = r
            
        gaze_map = runner._run_autogaze(frames)[0].cpu().float().numpy()
        
        for col in range(T):
            ax = axes[row, col]
            ax.imshow(frames[col])
            ax.imshow(gaze_map[col], cmap='jet', alpha=0.4, extent=[0, 224, 224, 0])
            if col == 0: ax.set_ylabel(f"Ratio: {r}", fontsize=12, fontweight='bold')
            ax.axis('off')
            
    runner.gazing_ratio = orig_ratio
    plt.tight_layout()
    plt.show()

# Example:
# plot_gaze_sweep(res_nvila['runner'], frames)